In [ ]:
BATCH_SIZE = 8
NUM_HEADS = 16 
SEQ_LEN = 4
HEAD_DIM = 3

In [2]:
import numpy as np 

Q = np.arange(BATCH_SIZE * NUM_HEADS * SEQ_LEN * HEAD_DIM).reshape(BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM)
K = np.arange(BATCH_SIZE * NUM_HEADS * SEQ_LEN * HEAD_DIM).reshape(BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM)
V = np.arange(BATCH_SIZE * NUM_HEADS * SEQ_LEN * HEAD_DIM).reshape(BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM)


In [7]:
import torch


BLOCK_SIZE_Q = 64

grid = (
    SEQ_LEN // BLOCK_SIZE_Q,
    BATCH_SIZE * NUM_HEADS,
    1,
)

grid


(64, 128, 1)

In [8]:
import numpy as np

BATCH_SIZE = 8
NUM_HEADS = 16
SEQ_LEN = 4096
BLOCK_SIZE_Q = 64

grid_x = SEQ_LEN // BLOCK_SIZE_Q
grid_y = BATCH_SIZE * NUM_HEADS

for pid_x in range(2):  # just first 2 blocks
    for pid_y in range(2):  # just first 2 heads
        q_start = pid_x * BLOCK_SIZE_Q
        q_end = q_start + BLOCK_SIZE_Q
        
        batch = pid_y // NUM_HEADS
        head = pid_y % NUM_HEADS
        
        print(f"Program ({pid_x}, {pid_y})")
        print(f"  Handles batch={batch}, head={head}")
        print(f"  Handles query rows {q_start} → {q_end-1}")
        print()

Program (0, 0)
  Handles batch=0, head=0
  Handles query rows 0 → 63

Program (0, 1)
  Handles batch=0, head=1
  Handles query rows 0 → 63

Program (1, 0)
  Handles batch=0, head=0
  Handles query rows 64 → 127

Program (1, 1)
  Handles batch=0, head=1
  Handles query rows 64 → 127



In [9]:
import numpy as np

S = 4
D = 3

np.random.seed(0)
Q = np.random.randn(S, D)
K = np.random.randn(S, D)
V = np.random.randn(S, D)

O = np.zeros((S, D))

for i in range(S):  # each Q row
    m = -np.inf
    l = 0.0
    o = np.zeros(D)

    print(f"\nProcessing Q row {i}")
    
    for j in range(S):  # streaming over K rows
        
        score = Q[i] @ K[j]

        m_new = max(m, score)
        
        l = l * np.exp(m - m_new) + np.exp(score - m_new)
        o = o * np.exp(m - m_new) + np.exp(score - m_new) * V[j]
        
        m = m_new

        print(f"  step (i={i}, j={j})")
        print(f"  score = {score:.4f}")
        print(f"  running max = {m:.4f}")
        print(f"  running l = {l:.4f}")
        print(f"  partial output = {o}")

    O[i] = o / l

print("\nFinal Output:")
print(O)


Processing Q row 0
  step (i=0, j=0)
  score = 1.8256
  running max = 1.8256
  running l = 1.0000
  partial output = [ 2.26975462 -1.45436567  0.04575852]
  step (i=0, j=1)
  score = 0.9857
  running max = 1.8256
  running l = 1.4317
  partial output = [ 2.18894022 -0.79260642  0.68013675]
  step (i=0, j=2)
  score = -2.2882
  running max = 1.8256
  running l = 1.4481
  partial output = [ 2.19147282 -0.78642538  0.66562596]
  step (i=0, j=3)
  score = 0.7725
  running max = 1.8256
  running l = 1.7969
  partial output = [ 1.50045081 -0.90779825  0.72016997]

Processing Q row 1
  step (i=1, j=0)
  score = 1.4989
  running max = 1.4989
  running l = 1.0000
  partial output = [ 2.26975462 -1.45436567  0.04575852]
  step (i=1, j=1)
  score = 3.7385
  running max = 3.7385
  running l = 1.1065
  partial output = [0.05453716 1.3778943  1.47423189]
  step (i=1, j=2)
  score = 1.6015
  running max = 3.7385
  running l = 1.2245
  partial output = [0.07282141 1.42251858 1.3694706 ]
  step (i=1, 